# 🤖 Day 20: Agentic AI & LLM Foundations (Hinglish)

Welcome to the Day 20 Practical Notebook! Aaj hum log concepts ko code ke through explore karenge. Is notebook mein:
1. **ReAct (Reason + Act) Agent Loop Simulation**
2. **Tokenization & Vocabulary Mapping**
3. **Embedding Vector Cosine Similarity**
4. **Temperature Logits Scaling & Sampling**

Chaliye shuru karte hain!

--- 
## 🛠️ 1. ReAct (Reason + Act) Loop Simulation

ReAct Prompting mein Agent pehle sochta hai (**Thought**), fir action leta hai (**Action**), fir result dekhta hai (**Observation**), aur process ko repeat karta hai jab tak goal achieve na ho jaye.

Hum ek mock Agent banayenge jiske paas do tools hain:
- `get_weather(city)`: City ka temperature return karta hai.
- `calculate_math(expression)`: Math expressions compute karta hai.

In [ ]:
import time

# Mock Tools definition
def get_weather(city):
    """Mock weather API tool"""
    city = city.lower().strip()
    weather_data = {
        "jaipur": 38,
        "delhi": 42,
        "mumbai": 32,
        "goa": 30
    }
    return f"Temperature in {city.capitalize()} is {weather_data.get(city, 'unknown')}°C"

def calculate_math(expression):
    """Mock calculator tool"""
    try:
        # Safety check: only allow numbers and operators
        allowed = "0123456789+-*/(). "
        if all(c in allowed for c in expression):
            return str(eval(expression))
        else:
            return "Error: Invalid characters"
    except Exception as e:
        return f"Error calculation: {str(e)}"

# Agent Execution Loop
class MockReActAgent:
    def __init__(self):
        self.tools = {
            "get_weather": get_weather,
            "calculate_math": calculate_math
        }
    
    def run(self, goal):
        print(f"🎯 Goal: {goal}\n")
        
        # Planning & Reasoning steps
        steps = [
            {
                "thought": "Mujhe Jaipur ka temperature check karna hoga aur usmein 5 degrees add karna hai. Pehle 'get_weather' tool call karta hoon for 'Jaipur'.",
                "action": "get_weather",
                "action_input": "jaipur"
            },
            {
                "thought": "Jaipur ka temperature 38 mil gaya hai. Ab mujhe calculation karni hai: 38 + 5. Iske liye 'calculate_math' use karunga.",
                "action": "calculate_math",
                "action_input": "38 + 5"
            }
        ]
        
        for i, step in enumerate(steps):
            print(f"🤔 Step {i+1} - Thought: {step['thought']}")
            time.sleep(1)
            
            tool_name = step["action"]
            tool_input = step["action_input"]
            tool_func = self.tools[tool_name]
            
            print(f"🎬 Action: Calling {tool_name} with '{tool_input}'...")
            observation = tool_func(tool_input)
            time.sleep(1)
            
            print(f"👀 Observation: {observation}\n")
            
        # Reflection and final answer
        final_thought = "Jaipur ka temp 38°C hai aur 5 degrees add karne par 43 hota hai. Task complete ho gaya hai."
        print(f"🧐 Reflection: {final_thought}")
        print(f"🏁 Final Response: Jaipur ka naya temperature 43°C hoga.")

# Run the simulation
agent = MockReActAgent()
agent.run("Find the temperature of Jaipur and add 5 to it.")

--- 
## 🔠 2. Tokenization & Vocabulary Mapping

Computers words ko nahi samajhte. Wo text ko numbers (Token IDs) mein split karte hain using a **Vocabulary**.
Yahan hum ek basic **Word Tokenizer** aur **Vocabulary Encoder** banayenge.

In [ ]:
# Sample Sentence
sentence = "I love Machine Learning and Agentic AI"

# 1. Tokenization: Text ko tokens (words) mein split karna
tokens = sentence.split()
print("📝 Step 1 - Tokens:", tokens)

# 2. Vocabulary creation (Mapping each unique token to an ID)
vocab = {word: idx for idx, word in enumerate(sorted(list(set(tokens))))}
# Special token padding / unknown
vocab["<UNK>"] = len(vocab)
print("📚 Step 2 - Vocabulary Mapping:")
for k, v in vocab.items():
    print(f"  '{k}': {v}")

# 3. Encoding: Text to Token IDs
encoded = [vocab.get(token, vocab["<UNK>"]) for token in tokens]
print("\n🔢 Step 3 - Encoded Tokens (IDs):", encoded)

# 4. Decoding: IDs back to Text
inv_vocab = {v: k for k, v in vocab.items()}
decoded = [inv_vocab[idx] for idx in encoded]
print("🗣️ Step 4 - Decoded back to sentence:", " ".join(decoded))

--- 
## 📐 3. Embedding Vector Cosine Similarity

Embeddings are high-dimensional vectors representing word meanings. Semantically similar words (jaise Cat aur Dog) ke vector direction same hoti hai, isliye unka **Cosine Similarity** score high hota hai.

$$\text{Cosine Similarity} = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|}$$

Hum Numpy use karke check karenge similarities.

In [ ]:
import numpy as np

# Mock 3-Dimensional Embeddings
embeddings = {
    "cat": np.array([0.9, 0.1, 0.0]),
    "dog": np.array([0.85, 0.15, 0.0]),
    "apple": np.array([0.1, 0.8, 0.5]),
    "banana": np.array([0.15, 0.75, 0.6])
}

def cosine_similarity(v1, v2):
    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    return dot_product / (norm_v1 * norm_v2)

print("📊 Checking Cosine Similarities between Embeddings:")
print(f"- Similarity(Cat, Dog): {cosine_similarity(embeddings['cat'], embeddings['dog']):.4f} (High Similarity - both are pets)")
print(f"- Similarity(Apple, Banana): {cosine_similarity(embeddings['apple'], embeddings['banana']):.4f} (High Similarity - both are fruits)")
print(f"- Similarity(Cat, Apple): {cosine_similarity(embeddings['cat'], embeddings['apple']):.4f} (Low Similarity - animals vs fruits)")

--- 
## 🎛️ 4. Temperature, Logits Scaling & Sampling

Model probability outputs (logits) ko temperature se divide kiya jata hai. 
- **Low Temperature (e.g. 0.1)** makes the distribution sharp (high chance for the top word, low creativity).
- **High Temperature (e.g. 1.5)** flattens the distribution (gives other words a chance, high creativity).

In [ ]:
import numpy as np

def softmax(logits):
    exp_logits = np.exp(logits - np.max(logits))
    return exp_logits / exp_logits.sum()

# Suppose model predictions (logits) for the next word of "I love python because it is ..."
words = ["easy", "fast", "powerful", "banana"]
logits = np.array([4.0, 3.0, 2.5, -1.0])  # "easy" has highest logit

def sample_next_word(logits, temperature=1.0):
    # Scale logits by temperature
    scaled_logits = logits / temperature
    probabilities = softmax(scaled_logits)
    
    # Select word based on probability distribution
    sampled_idx = np.random.choice(len(words), p=probabilities)
    return words[sampled_idx], probabilities

print("🔥 Temperature Effects:")
for temp in [0.1, 0.7, 1.5]:
    sampled, probs = sample_next_word(logits, temp)
    print(f"\n🌡️ Temperature = {temp}:")
    for word, prob in zip(words, probs):
        print(f"  '{word}': {prob*100:.2f}%")
    print(f"  👉 Sampled Word: '{sampled}'")